# PD-L1 Stomach — Flat 5-class Positive-weighted Detection

일반 YOLOv11-m으로 5개 클래스를 직접 검출합니다.
계층형 root/branch 없이 Box + Cls + DFL loss를 사용합니다.

| ID | Class | 원본 라벨 |
|---:|---|---|
| 0 | Negative Epithelial | Negative Epithelial |
| 1 | Negative Immune | Negative Lymphocyte + Negative Macrophage |
| 2 | Positive Epithelial | Positive Epithelial |
| 3 | Positive Immune | Positive Lymphocyte + Positive Macrophage |
| 4 | other | other |

모든 세포 bbox를 유지하고 학습·검증·시각화에 동일한 클래스 매핑을 적용합니다.
현재 flat 4-class 노트북과 같은 데이터 split, augmentation 및 학습 설정을 사용합니다.

- GPU_ID=1, batch size=16, 입력 512×512, 최대 2000 epoch
- 원본당 epoch별 50회 random crop, AdamW(lr=1e-4), cosine scheduler
- crop의 80%에서 해당 이미지에 존재하는 원본 7개 클래스를 동일 가중치로 선택
- 검증: 슬라이드 3장 분리, 1024 이미지를 512 타일 4장으로 분할
- 매 epoch 검증, 10 epoch마다 예측 시각화와 학습 곡선 저장
- Best 기준: 0.7 × Macro F1 + 0.3 × Detection Recall

위에서부터 실행하세요. 다른 학습과 함께 실행한다면 GPU_ID를 확인하세요.
처음부터 학습하며 결과는 `../../model/mediautoAI_PDL1_Stomach/pdl1_flat_5class_positive_weighted/`에 저장합니다.

학습 Dataset에서 라벨이 0개인 crop은 같은 원본에서 다시 추출합니다(최대 32회, 실패 시 오류).
유효한 라벨이 전혀 없는 원본은 학습 Dataset 인덱스에서 제외하므로 빈 crop이 배치에 들어가지 않습니다.
검증 데이터는 필터링하지 않습니다. 일부만 라벨링된 crop의 미표기 세포는 이 변경으로 처리되지 않습니다.

## 양성 세포 가중치

복구본과 동일한 Dataset/crop/학습/검증 설정에서 분류 loss 가중치만 변경합니다.

| 클래스 | BCE positive-target 가중치 |
|---|---:|
| Negative Epithelial | 1.0 |
| Negative Immune | 1.0 |
| Positive Epithelial | 2.0 |
| Positive Immune | 2.0 |
| other | 1.0 |

첫 코드 셀의 `POSITIVE_EPITHELIAL_WEIGHT`, `POSITIVE_IMMUNE_WEIGHT`로 조절합니다.
`cls_pos_weight`는 해당 클래스의 positive-target BCE 항에 적용되며,
Box/DFL이나 분류 loss 전체를 2배로 만드는 설정은 아닙니다.
가중치는 체크포인트의 `params`에 저장됩니다.
위에서부터 실행하세요. GPU_ID=1이며, 처음부터 학습하도록 설정되어 있습니다.


In [ ]:
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm
import yaml

from nets import nn
from utils import util
from utils.ihc_dataset import (
    IHCHierarchicalDataset,
    PDL1_CLASS_NAMES,
    count_record_classes,
    discover_records,
    slide_id_from_record,
    split_records_by_slide_balanced,
)
from utils.valid import (
    compute_point_label_metrics_single,
    plot_training_progress,
    visualize_ground_truth_and_prediction_separately_detail_single,
)

warnings.filterwarnings('ignore')
GPU_ID = 0
if torch.cuda.is_available():
    if GPU_ID >= torch.cuda.device_count():
        raise RuntimeError(
            f'GPU_ID={GPU_ID}, available GPUs={torch.cuda.device_count()}'
        )
    torch.cuda.set_device(GPU_ID)
    device = torch.device(f'cuda:{GPU_ID}')
    print(f'device: {device} | {torch.cuda.get_device_name(GPU_ID)}')
else:
    device = torch.device('cpu')
    print('device: cpu')

with open('utils/detail_args.yaml', errors='ignore') as file:
    params = yaml.safe_load(file)

CLASS_NAMES = {0: 'Negative Epithelial', 1: 'Negative Immune', 2: 'Positive Epithelial', 3: 'Positive Immune', 4: 'other'}
CLASS_COLORS = {0: '#1f77b4', 1: '#2ca02c', 2: '#d62728', 3: '#ff7f0e', 4: '#7f7f7f'}
SOURCE_TO_FLAT = (0, 1, 1, 2, 3, 3, 4)
CLASS_MAPPING = {'Negative Epithelial': ['Negative Epithelial'], 'Negative Immune': ['Negative Lymphocyte', 'Negative Macrophage'], 'Positive Epithelial': ['Positive Epithelial'], 'Positive Immune': ['Positive Lymphocyte', 'Positive Macrophage'], 'other': ['other']}
params['names'] = CLASS_NAMES

# 클래스 순서: Negative Epithelial / Negative Immune /
#              Positive Epithelial / Positive Immune / other
POSITIVE_EPITHELIAL_WEIGHT = 2.0
POSITIVE_IMMUNE_WEIGHT = 2.0
params['cls_pos_weight'] = [
    1.0, 1.0, POSITIVE_EPITHELIAL_WEIGHT, POSITIVE_IMMUNE_WEIGHT, 1.0,
]
print('Classification positive-target weights:')
for class_id, name in CLASS_NAMES.items():
    print(f"  {name:24s}: {params['cls_pos_weight'][class_id]:.2f}")


In [ ]:
class Flat5ClassDataset(torch.utils.data.Dataset):
    """Map seven source labels to 5 flat classes, retaining every bounding box."""

    class_names = CLASS_NAMES
    class_colors = CLASS_COLORS

    def __init__(self, base_dataset, require_labels=False, max_crop_attempts=32):
        self.base_dataset = base_dataset
        self.require_labels = bool(require_labels)
        self.max_crop_attempts = int(max_crop_attempts)
        if self.max_crop_attempts < 1:
            raise ValueError('max_crop_attempts must be positive')
        self.record_indices = list(range(len(base_dataset)))
        if self.require_labels:
            if base_dataset.four_crop or not base_dataset.random_crop:
                raise ValueError('require_labels is intended for random-crop training only')
            # 원본에 유효한 라벨이 전혀 없으면 재추출해도 소용없으므로 제외합니다.
            self.record_indices = [
                i for i in self.record_indices
                if len(base_dataset._get_resized_pair(i)[1]) > 0
            ]
            if not self.record_indices:
                raise ValueError('유효한 라벨이 있는 학습 원본이 없습니다.')
        self.excluded_empty_records = len(base_dataset) - len(self.record_indices)

    def __len__(self):
        return len(self.record_indices)

    def __getitem__(self, index):
        record_index = self.record_indices[index]
        attempts = self.max_crop_attempts if self.require_labels else 1
        for _ in range(attempts):
            image, classes, boxes, indices = self.base_dataset[record_index]
            if not self.require_labels or classes.numel() > 0:
                break
        else:
            # 무한 재시도를 피하고 라벨/crop 문제를 명시적으로 알립니다.
            raise RuntimeError(
                f'원본 index={record_index}: {attempts}회 crop 후에도 라벨이 없습니다. '
                '원본 라벨 좌표와 crop 설정을 확인하세요.'
            )
        mapping = classes.new_tensor(SOURCE_TO_FLAT)
        flat_classes = mapping[classes.long()]
        return image, flat_classes, boxes, indices

    @property
    def cache_memory_bytes(self):
        return self.base_dataset.cache_memory_bytes


def collate_fn(batch):
    return IHCHierarchicalDataset.collate_fn(batch)


In [ ]:
input_size = 512
resize_size = 1024
data_root = '../../data/mediautoAI_PDL1_Stomach/'
VALIDATION_SLIDE_COUNT = 3

records = discover_records(data_root)
train_records, val_records = split_records_by_slide_balanced(
    records, val_slide_count=VALIDATION_SLIDE_COUNT
)
train_slides = {slide_id_from_record(record) for record in train_records}
val_slides = {slide_id_from_record(record) for record in val_records}
assert train_slides.isdisjoint(val_slides)

# 80%는 이미지에 존재하는 원본 클래스 중 하나를 동일 가중치로 선택합니다.
# 나머지는 일반 random crop입니다. 현재 flat 4-class 설정과 동일합니다.
CLASS_AWARE_CROP_PROBABILITY = 0.8
CROP_CLASS_WEIGHTS = [1, 1, 1, 1, 1, 1, 1]

train_base = IHCHierarchicalDataset(
    train_records, input_size=input_size, resize_size=resize_size,
    augment=True, random_crop=True, cache_images=True,
    class_aware_crop_probability=CLASS_AWARE_CROP_PROBABILITY,
    class_sampling_weights=CROP_CLASS_WEIGHTS,
)
val_base = IHCHierarchicalDataset(
    val_records, input_size=input_size, resize_size=resize_size,
    augment=False, random_crop=False, cache_images=True, four_crop=True,
)
train_dataset = Flat5ClassDataset(train_base, require_labels=True)
print(f'Excluded empty train source patches: {train_dataset.excluded_empty_records}')
val_dataset = Flat5ClassDataset(val_base)


In [ ]:
source_counts = count_record_classes(train_records).astype(np.int64)
val_source_counts = count_record_classes(val_records).astype(np.int64)
train_counts = np.zeros(len(CLASS_NAMES), dtype=np.int64)
val_counts = np.zeros(len(CLASS_NAMES), dtype=np.int64)
for source_id, class_id in enumerate(SOURCE_TO_FLAT):
    train_counts[class_id] += source_counts[source_id]
    val_counts[class_id] += val_source_counts[source_id]
assert train_counts.sum() == source_counts.sum()
assert val_counts.sum() == val_source_counts.sum()
print('Flat 5-class label counts:')
for class_id, class_name in CLASS_NAMES.items():
    print(f'  [{class_id}] {class_name:24s}: train={train_counts[class_id]:,} | val={val_counts[class_id]:,}')


In [ ]:
EPOCH_DATA_REPEATS = 50
batch_size = 16
train_epoch_dataset = torch.utils.data.ConcatDataset(
    [train_dataset] * EPOCH_DATA_REPEATS
)
train_loader = torch.utils.data.DataLoader(
    train_epoch_dataset, batch_size=batch_size, shuffle=True, num_workers=4,
    collate_fn=collate_fn, drop_last=False,
    pin_memory=(device.type == 'cuda'), persistent_workers=True,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=4,
    collate_fn=collate_fn, drop_last=False,
    pin_memory=(device.type == 'cuda'), persistent_workers=True,
)

print(f'\nTrain source patches : {len(train_dataset):,} ({len(train_slides)} slides)')
print(f'Samples/epoch        : {len(train_epoch_dataset):,}')
print(f'Train batches        : {len(train_loader):,}')
print(f'Val source/tiles     : {len(val_records)} / {len(val_dataset)}')
cache_mib = (train_dataset.cache_memory_bytes + val_dataset.cache_memory_bytes) / 1024**2
print(f'1024 image cache     : {cache_mib:.1f} MiB')


In [ ]:
# 매 호출마다 random crop이 실제로 바뀌는지 간단히 확인합니다.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for axis in axes:
    image, classes, boxes, _ = train_dataset[0]
    axis.imshow(image.permute(1, 2, 0).numpy())
    for class_id, box in zip(classes.long(), boxes):
        color = CLASS_COLORS[int(class_id)]
        axis.scatter(
            box[0].item() * input_size, box[1].item() * input_size,
            facecolors='none', edgecolors=color, s=18, linewidths=1,
        )
    axis.set_title('\n'.join(
        f'{CLASS_NAMES[c]}: {(classes == c).sum().item()}' for c in CLASS_NAMES
    ), fontsize=9)
    axis.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

LEARNING_RATE = 1e-4
MIN_LEARNING_RATE = 1e-6
MAX_EPOCHS = 2000

model = nn.yolo_v11_m(num_classes=len(CLASS_NAMES)).to(device)
optimizer = torch.optim.AdamW(
    util.set_params(model, params['weight_decay']),
    lr=LEARNING_RATE, betas=(0.9, 0.999),
    weight_decay=params['weight_decay'],
)
criterion = util.ComputeLoss(model, params)
scheduler = CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE
)

model.train()
smoke_images, smoke_targets = next(iter(train_loader))
smoke_images = smoke_images.to(device).float() / 255.0
with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
    smoke_outputs = model(smoke_images)
    smoke_box, smoke_cls, smoke_dfl = criterion(smoke_outputs, smoke_targets)
print(
    f'Loss smoke test — Box={smoke_box.item():.4f} | '
    f'Cls={smoke_cls.item():.4f} | DFL={smoke_dfl.item():.4f}'
)
del smoke_images, smoke_targets, smoke_outputs
del smoke_box, smoke_cls, smoke_dfl
if device.type == 'cuda':
    torch.cuda.empty_cache()

print(f'Model: flat YOLOv11-m, classes={len(CLASS_NAMES)}')
print(f'Epochs={MAX_EPOCHS}, data repeats/epoch={EPOCH_DATA_REPEATS}')
print('Early stopping: disabled')


In [ ]:
train_losses = []
val_det_recalls = []
val_cls_accs = []
val_macro_precisions = []
val_macro_recalls = []
val_macro_f1s = []
val_class_stats_history = []

save_dir = '../../model/mediautoAI_PDL1_Stomach/pdl1_flat_5class_positive_weighted/'
os.makedirs(save_dir, exist_ok=True)
start_epoch = 0
best_validation_score = 0.0
accumulate = max(round(64 / batch_size), 1)
amp_scale = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
VALIDATION_CONFIDENCE_THRESHOLD = 0.10

# 재개할 때 아래 블록의 주석을 해제합니다.
# checkpoint_path = os.path.join(save_dir, 'last_model.pt')
# if os.path.exists(checkpoint_path):
#     checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
#     if checkpoint.get('architecture') != 'pdl1_flat_5class':
#         raise ValueError('Flat 5-class checkpoint가 아닙니다.')
#     if checkpoint.get('params', {}).get('cls_pos_weight') != params['cls_pos_weight']:
#         raise ValueError('체크포인트와 현재 클래스 가중치가 다릅니다.')
#     model.load_state_dict(checkpoint['model_state_dict'])
#     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#     scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
#     amp_scale.load_state_dict(checkpoint['amp_scale_state_dict'])
#     start_epoch = checkpoint['epoch'] + 1
#     best_validation_score = checkpoint.get('best_validation_score', 0.0)

print(f'Gradient accumulation steps: {accumulate}')

for epoch in range(start_epoch, MAX_EPOCHS):
    model.train()
    avg_box = util.AverageMeter()
    avg_cls = util.AverageMeter()
    avg_dfl = util.AverageMeter()
    avg_total = util.AverageMeter()
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(enumerate(train_loader), total=len(train_loader))
    for step, (images, targets) in progress:
        images = images.to(device, non_blocking=True).float() / 255.0
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss_box, loss_cls, loss_dfl = criterion(outputs, targets)
            unscaled_loss = loss_box + loss_cls + loss_dfl
            loss = unscaled_loss / accumulate

        batch_n = images.size(0)
        avg_box.update(loss_box.item(), batch_n)
        avg_cls.update(loss_cls.item(), batch_n)
        avg_dfl.update(loss_dfl.item(), batch_n)
        avg_total.update(unscaled_loss.item(), batch_n)
        amp_scale.scale(loss).backward()
        if (step + 1) % accumulate == 0 or (step + 1) == len(train_loader):
            amp_scale.step(optimizer)
            amp_scale.update()
            optimizer.zero_grad(set_to_none=True)

        memory = (
            f'{torch.cuda.memory_reserved() / 1e9:.3f}G'
            if device.type == 'cuda' else 'CPU'
        )
        progress.set_description(
            f'Epoch {epoch + 1}/{MAX_EPOCHS} | Memory {memory} | '
            f'Box {avg_box.avg:.3f} | Cls {avg_cls.avg:.3f} | DFL {avg_dfl.avg:.3f}'
        )

    scheduler.step()
    train_losses.append(avg_total.avg)

    try:
        metrics = compute_point_label_metrics_single(
            model, val_loader, device, params, distance_threshold=16,
            class_agnostic_nms=True,
            confidence_threshold=VALIDATION_CONFIDENCE_THRESHOLD,
        )
        detection_recall = float(metrics.get('detection_recall', 0))
        cls_accuracy = float(metrics.get('classification_accuracy', 0))
        macro_precision = float(metrics.get('macro_precision', 0))
        macro_recall = float(metrics.get('macro_recall', 0))
        macro_f1 = float(metrics.get('macro_f1', 0))
        class_stats = metrics.get('class_stats', {})
        validation_score = 0.7 * macro_f1 + 0.3 * detection_recall
    except Exception as error:
        print(f'Validation 오류: {error}')
        detection_recall = cls_accuracy = 0.0
        macro_precision = macro_recall = macro_f1 = 0.0
        validation_score = 0.0
        class_stats = {}

    val_det_recalls.append(detection_recall)
    val_cls_accs.append(cls_accuracy)
    val_macro_precisions.append(macro_precision)
    val_macro_recalls.append(macro_recall)
    val_macro_f1s.append(macro_f1)
    val_class_stats_history.append(class_stats)

    print(f'\nEpoch {epoch + 1}/{MAX_EPOCHS}')
    print(f'  Detection Recall : {detection_recall:.4f}')
    print(f'  Class Accuracy   : {cls_accuracy:.4f}')
    print(f'  Macro P/R/F1     : {macro_precision:.4f} / {macro_recall:.4f} / {macro_f1:.4f}')
    for name, stats in class_stats.items():
        print(
            f"  {name:24s} P={stats['precision']:.4f} | "
            f"R={stats['recall']:.4f} | F1={stats['f1']:.4f}"
        )

    improved = validation_score > best_validation_score
    if improved:
        best_validation_score = validation_score

    checkpoint = {
        'architecture': 'pdl1_flat_5class',
        'experiment': 'positive_weighted',
        'require_labels_train': True,
        'class_mapping': CLASS_MAPPING,
        'class_names': CLASS_NAMES,
        'source_to_flat': SOURCE_TO_FLAT,
        'num_classes': len(CLASS_NAMES),
        'epoch': epoch,
        'epoch_data_repeats': EPOCH_DATA_REPEATS,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'amp_scale_state_dict': amp_scale.state_dict(),
        'train_loss': avg_total.avg,
        'box_loss': avg_box.avg,
        'cls_loss': avg_cls.avg,
        'dfl_loss': avg_dfl.avg,
        'detection_recall': detection_recall,
        'classification_accuracy': cls_accuracy,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'validation_score': validation_score,
        'best_validation_score': best_validation_score,
        'class_stats': class_stats,
        'params': params,
    }
    torch.save(checkpoint, os.path.join(save_dir, 'last_model.pt'))
    if improved:
        torch.save(checkpoint, os.path.join(save_dir, 'best_model.pt'))
        print(f'  New best validation score: {validation_score:.4f}')

    if (epoch + 1) % 10 == 0:
        try:
            sample_index = random.randrange(len(val_dataset))
            visualize_ground_truth_and_prediction_separately_detail_single(
                model, val_dataset, idx=sample_index, epoch=epoch + 1,
                save_dir=save_dir, class_names=CLASS_NAMES,
                class_colors=CLASS_COLORS,
                conf_threshold=VALIDATION_CONFIDENCE_THRESHOLD,
            )
            plot_training_progress(
                train_losses, val_det_recalls, val_cls_accs,
                val_macro_precisions, val_macro_recalls, val_macro_f1s,
                epoch + 1, save_dir,
                class_stats_history=val_class_stats_history,
            )
        except Exception as error:
            print(f'시각화 오류: {error}')

print('Flat 5-class 학습 완료')
print(f'Best validation score: {best_validation_score:.4f}')
print(f'Save directory: {save_dir}')
